# CS781 Phase 1 — Model 4 — Generate Codabench Submission

**Inputs to attach**: (1) the tokenizer dataset, (2) model 4's training notebook output (needs its `legal-gqamoe-final` folder with the final saved checkpoint), and (3) optionally, the Prepare Test Set notebook's output -- same Phase 1 Task 1 test set as model 1 used, so that snapshot is reusable here unchanged.

Needs internet + HF login only as a fallback (if no snapshot is attached). No GPU-heavy work here compared to training -- no backward pass, and KV-caching makes each of the 128 decode steps cheap after the prompt prefill.

## 1. Setup

In [ ]:
import os

## 2. Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 3. Configuration

In [ ]:
import glob, json, math, time, zipfile

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"  # verified via list_repo_files() -- see model 1's notebook history

TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 256
TEMPERATURE = 0.7
TOP_P = 0.95
GENERATION_BATCH_SIZE = 32

OUTPUT_DIR = "/kaggle/working/cs781_phase1_model4_submission"
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 4. Load Test Data

Prefers an attached snapshot from the Prepare Test Set notebook over downloading fresh -- the dataset repo notes the test file is dynamic and can change. HF login only happens in the fallback branch below, so it's never needed (and never asked for) when a snapshot is already attached, which is the normal case.

In [ ]:
def find_dirs(filename):
    return sorted({os.path.dirname(p) for p in glob.glob(f"/kaggle/input/**/{filename}", recursive=True)})

snapshot_dirs = find_dirs("test_snapshot.json")
if snapshot_dirs:
    test_file_path = os.path.join(snapshot_dirs[0], "test_snapshot.json")
    print("Using attached snapshot:", test_file_path)
    stats_path = os.path.join(snapshot_dirs[0], "test_stats.json")
    if os.path.exists(stats_path):
        print("Snapshot info:", json.load(open(stats_path)))
else:
    from huggingface_hub import login, hf_hub_download
    from kaggle_secrets import UserSecretsClient
    print("No snapshot attached -- logging in and downloading fresh (this will be whatever is live right now).")
    HF_TOKEN = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
    test_file_path = hf_hub_download(
        repo_id=HF_DATASET_REPO, filename=TEST_FILE, repo_type="dataset", token=HF_TOKEN
    )
    print("Downloaded:", test_file_path)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)
print("Number of test examples:", len(test_data))

## 5. Load Tokenizer

In [ ]:
from transformers import AutoTokenizer

cands = [d for d in find_dirs("tokenizer.json") if "checkpoint-" not in d and "legal-gqamoe-final" not in d]
if not cands:
    raise FileNotFoundError("No tokenizer dataset found -- attach it via Add Input.")
TOKENIZER_DIR = ([d for d in cands if os.path.exists(os.path.join(d, "trained_on.json"))] or cands)[0]

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR)
tokenizer.padding_side = "left"  # required for correct batched generation
print("Tokenizer:", TOKENIZER_DIR, "| vocab size:", len(tokenizer))

## 5b. Load Pre-Tokenized Test Set (if attached)

In [ ]:
tok_cache_dirs = find_dirs("test_tokenized.json")
test_token_ids = None
if tok_cache_dirs:
    tok_cache_path = os.path.join(tok_cache_dirs[0], "test_tokenized.json")
    with open(tok_cache_path) as f:
        test_token_ids = json.load(f)  # {example_id: [token ids]}
    print("Using pre-tokenized cache:", tok_cache_path, f"({len(test_token_ids)} examples)")
else:
    print("No pre-tokenized cache attached -- will tokenize prompts inline during generation.")

## 6. Define The Model

Three files, model 4's exact tested architecture, unchanged: `legal_gqa_model.py` (model 1's tested module, supplying RMSNorm and `GroupedQueryAttention` -- reused for both the MHA and GQA layers, just instantiated with a different n_kv_head), `moe.py` (model 2's tested MoE module), `model.py`. Written to real files and imported for the same reason as every training notebook: `transformers` reads `__file__` off a model class's module.

In [ ]:
%%writefile /kaggle/working/legal_gqa_model.py
"""
Model 1 for CS781 Phase-1: GQA + RoPE + SwiGLU + RMSNorm decoder-only transformer.

Wrapped as a transformers PretrainedConfig/PreTrainedModel subclass so it is a
drop-in replacement for GPT2Config/GPT2LMHeadModel in the starter notebook:
same save_pretrained/from_pretrained, same Trainer + TrainingArguments +
resume_from_checkpoint behavior, same model.generate() call signature.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint
from transformers import PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast


class LegalGQAConfig(PretrainedConfig):
    model_type = "legal_gqa"

    def __init__(
        self,
        vocab_size=50257,
        n_positions=1024,
        n_embd=1024,
        n_layer=8,
        n_head=16,
        n_kv_head=4,
        d_ff=2688,
        rope_theta=10000.0,
        dropout=0.1,
        initializer_range=0.02,
        rms_norm_eps=1e-5,
        bos_token_id=50256,
        eos_token_id=50256,
        pad_token_id=50256,
        activation_checkpointing=False,
        tie_word_embeddings=True,
        **kwargs,
    ):
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        assert n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"
        self.vocab_size = vocab_size
        self.n_positions = n_positions
        self.n_embd = n_embd
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.d_ff = d_ff
        self.rope_theta = rope_theta
        self.dropout = dropout
        self.initializer_range = initializer_range
        self.rms_norm_eps = rms_norm_eps
        self.use_cache = True
        # own implementation (HF's gradient_checkpointing flag is not supported here)
        self.activation_checkpointing = activation_checkpointing
        super().__init__(
            bos_token_id=bos_token_id,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
            tie_word_embeddings=tie_word_embeddings,
            **kwargs,
        )


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(q, k, cos, sin):
    # q, k: (B, n_head, T, head_dim); cos/sin: (T, head_dim)
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    q_rot = (q * cos) + (rotate_half(q) * sin)
    k_rot = (k * cos) + (rotate_half(k) * sin)
    return q_rot, k_rot


class GroupedQueryAttention(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.n_head = config.n_head
        self.n_kv_head = config.n_kv_head
        self.head_dim = config.n_embd // config.n_head
        self.n_rep = self.n_head // self.n_kv_head
        self.dropout_p = config.dropout

        self.q_proj = nn.Linear(config.n_embd, config.n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.n_head * self.head_dim, config.n_embd, bias=False)

    def forward(self, x, cos, sin, attn_bias, is_causal, past_key_value=None, use_cache=False):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        q, k = apply_rope(q, k, cos, sin)
        # RoPE tables are fp32, v comes out of a linear (fp16 under autocast): force one dtype
        q, k = q.to(v.dtype), k.to(v.dtype)

        if past_key_value is not None:
            past_k, past_v = past_key_value
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        present = (k, v) if use_cache else None

        # GQA: broadcast each KV head across its group of query heads
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_bias,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=is_causal,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_head * self.head_dim)
        return self.o_proj(out), present


class SwiGLU(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.gate_proj = nn.Linear(config.n_embd, config.d_ff, bias=False)
        self.up_proj = nn.Linear(config.n_embd, config.d_ff, bias=False)
        self.down_proj = nn.Linear(config.d_ff, config.n_embd, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class DecoderBlock(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.ln_1 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.attn = GroupedQueryAttention(config)
        self.ln_2 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.ffn = SwiGLU(config)

    def forward(self, x, cos, sin, attn_bias, is_causal, past_key_value=None, use_cache=False):
        attn_out, present = self.attn(
            self.ln_1(x), cos, sin, attn_bias, is_causal, past_key_value, use_cache
        )
        x = x + attn_out
        x = x + self.ffn(self.ln_2(x))
        return x, present


class LegalGQAForCausalLM(PreTrainedModel):
    config_class = LegalGQAConfig
    # input embedding and output head share one weight; must be declared or
    # save_pretrained / Trainer checkpointing refuses to save
    _tied_weights_keys = {"lm_head.weight": "wte.weight"}

    def __init__(self, config: LegalGQAConfig):
        super().__init__(config)
        self.head_dim = config.n_embd // config.n_head

        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([DecoderBlock(config) for _ in range(config.n_layer)])
        self.ln_f = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # weight tying

        self.post_init()

    def _init_weights(self, module):
        std = self.config.initializer_range
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
        # scaled init on residual-stream output projections (GPT-2 recipe)
        for name, p in module.named_parameters(recurse=False):
            if name in ("o_proj.weight", "down_proj.weight"):
                nn.init.normal_(
                    p, mean=0.0, std=std / math.sqrt(2 * self.config.n_layer)
                )

    def get_input_embeddings(self):
        return self.wte

    def get_output_embeddings(self):
        return self.lm_head

    def set_output_embeddings(self, new_embeddings):
        self.lm_head = new_embeddings

    def set_input_embeddings(self, value):
        self.wte = value

    def _rope_slice(self, past_len, T, device, dtype):
        # Computed on the fly (float32, then cast). Deliberately NOT stored as a buffer:
        # non-persistent buffers come back uninitialized after from_pretrained().
        inv_freq = 1.0 / (
            self.config.rope_theta
            ** (torch.arange(0, self.head_dim, 2, device=device).float() / self.head_dim)
        )
        pos = torch.arange(past_len, past_len + T, device=device).float()
        emb = torch.cat((torch.outer(pos, inv_freq),) * 2, dim=-1)
        return emb.cos().to(dtype), emb.sin().to(dtype)

    def _build_attn_bias(self, attention_mask, past_len, T, device, dtype):
        """Returns (attn_bias, is_causal). attn_bias is None when the fast
        SDPA is_causal fused-kernel path is usable (no padding to mask)."""
        no_padding = attention_mask is None or bool(attention_mask.bool().all())
        if no_padding:
            return None, T > 1

        total_len = past_len + T
        pad = attention_mask[:, None, None, :].to(dtype)
        bias = (1.0 - pad) * torch.finfo(dtype).min

        if T > 1:
            q_pos = torch.arange(past_len, past_len + T, device=device).unsqueeze(1)
            k_pos = torch.arange(0, total_len, device=device).unsqueeze(0)
            causal = (k_pos > q_pos).to(dtype) * torch.finfo(dtype).min
            bias = bias + causal[None, None, :, :]
        return bias, False

    def forward(
        self,
        input_ids,
        attention_mask=None,
        past_key_values=None,
        labels=None,
        use_cache=None,
    ):
        # NOTE: no **kwargs here on purpose. If forward() accepts **kwargs, Trainer assumes the
        # model normalises its own loss, then multiplies it by n_gpu and skips the
        # gradient-accumulation division (loss reported 2x-64x too high).
        #
        # fp16 autocast is switched on HERE rather than relying on Trainer: under nn.DataParallel
        # on Kaggle the matmuls were measurably running in fp32 (no tensor cores, ~4x slower).
        # Only used for unpadded batches (all of training/eval); padded generation keeps the
        # exact fp32 path, since fp16 additive masks can overflow to -inf.
        amp = input_ids.is_cuda and (attention_mask is None or bool(attention_mask.bool().all()))
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            return self._forward_impl(input_ids, attention_mask, past_key_values, labels, use_cache)

    def _forward_impl(self, input_ids, attention_mask, past_key_values, labels, use_cache):
        use_cache = self.config.use_cache if use_cache is None else use_cache
        if self.training:
            use_cache = False  # never keep K/V during training
        B, T = input_ids.shape
        past_len = past_key_values[0][0].shape[2] if past_key_values is not None else 0

        x = self.drop(self.wte(input_ids))
        cos, sin = self._rope_slice(past_len, T, input_ids.device, x.dtype)
        attn_bias, is_causal = self._build_attn_bias(
            attention_mask, past_len, T, input_ids.device, x.dtype
        )

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            pkv = past_key_values[i] if past_key_values is not None else None
            if self.training and self.config.activation_checkpointing:
                x = torch_checkpoint(
                    lambda h, blk=block: blk(h, cos, sin, attn_bias, is_causal, None, False)[0],
                    x,
                    use_reentrant=False,
                )
            else:
                x, present = block(x, cos, sin, attn_bias, is_causal, pkv, use_cache)
                if use_cache:
                    presents.append(present)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        # when labels are given only the loss is needed; returning the full
        # (B, T, vocab) logits would just be gathered across GPUs for nothing
        return CausalLMOutputWithPast(
            loss=loss,
            logits=None if labels is not None else logits,
            past_key_values=tuple(presents) if use_cache else None,
        )

    # No prepare_inputs_for_generation / GenerationMixin here on purpose —
    # generation is done by generate_utils.generate_batch(), which calls
    # this model's forward() directly with its own tuple-based KV cache
    # rather than going through model.generate().

In [ ]:
%%writefile /kaggle/working/moe.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class Expert(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class MoE(nn.Module):
    def __init__(self, d_model, n_routed, n_shared, top_k, expert_d_ff, aux_loss_coef=0.01):
        super().__init__()
        self.n_routed, self.top_k = n_routed, top_k
        self.aux_loss_coef = aux_loss_coef
        self.router = nn.Linear(d_model, n_routed, bias=False)
        self.routed_experts = nn.ModuleList([Expert(d_model, expert_d_ff) for _ in range(n_routed)])
        self.shared_experts = nn.ModuleList([Expert(d_model, expert_d_ff) for _ in range(n_shared)])
        self.last_aux_loss = None

    def forward(self, x):
        B, T, D = x.shape
        flat = x.reshape(-1, D)                                 # (N, D), N = B*T
        N = flat.shape[0]

        logits = self.router(flat)                               # (N, n_routed)
        probs = F.softmax(logits, dim=-1)
        top_w, top_i = probs.topk(self.top_k, dim=-1)             # (N, top_k) each
        top_w = top_w / top_w.sum(dim=-1, keepdim=True)           # renormalize among selected

        out = torch.zeros_like(flat)
        for e, expert in enumerate(self.routed_experts):
            token_slot, k_slot = (top_i == e).nonzero(as_tuple=True)   # which tokens picked expert e, and at which of their top_k slots
            if token_slot.numel() == 0:
                continue
            expert_out = expert(flat[token_slot])                       # only THIS expert's assigned tokens, real compute saving
            out.index_add_(0, token_slot, expert_out * top_w[token_slot, k_slot].unsqueeze(-1))

        for expert in self.shared_experts:
            out = out + expert(flat)                                    # always on, every token

        if self.training:
            # Switch-Transformer style load-balancing aux loss: encourages the router
            # to spread tokens evenly across experts rather than collapsing onto a few.
            with torch.no_grad():
                assign_count = torch.zeros(self.n_routed, device=x.device)
                assign_count.scatter_add_(0, top_i.reshape(-1), torch.ones_like(top_i.reshape(-1), dtype=torch.float))
                frac_tokens = assign_count / (N * self.top_k)
            frac_prob = probs.mean(dim=0)
            self.last_aux_loss = self.aux_loss_coef * self.n_routed * (frac_tokens * frac_prob).sum()
        else:
            self.last_aux_loss = None

        return out.view(B, T, D)

In [ ]:
%%writefile /kaggle/working/model.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint
from transformers import PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast

from legal_gqa_model import RMSNorm, GroupedQueryAttention, LegalGQAConfig
from moe import MoE


class LegalGQAMoEConfig(PretrainedConfig):
    model_type = "legal_gqa_mha_moe"

    def __init__(
        self,
        vocab_size=50257, n_positions=1024, n_embd=1024, n_layer=9,
        mha_layers=(0, 4, 8),           # 0-indexed: layers 1, 5, 9 -- full MHA (n_kv_head = n_head)
        n_head=16, gqa_n_kv_head=4,     # the other 6 layers: GQA, same n_kv_head model 1/3 already proved
        n_routed_experts=6, n_shared_experts=1, moe_top_k=2, expert_d_ff=896, moe_aux_loss_coef=0.01,
        rope_theta=10000.0,
        dropout=0.0, initializer_range=0.02, rms_norm_eps=1e-5,
        bos_token_id=50256, eos_token_id=50256, pad_token_id=50256,
        activation_checkpointing=False, tie_word_embeddings=True,
        **kwargs,
    ):
        assert n_embd % n_head == 0
        self.vocab_size, self.n_positions, self.n_embd, self.n_layer = vocab_size, n_positions, n_embd, n_layer
        self.mha_layers = list(mha_layers)
        self.n_head, self.gqa_n_kv_head = n_head, gqa_n_kv_head
        self.n_routed_experts, self.n_shared_experts = n_routed_experts, n_shared_experts
        self.moe_top_k, self.expert_d_ff, self.moe_aux_loss_coef = moe_top_k, expert_d_ff, moe_aux_loss_coef
        self.rope_theta = rope_theta
        self.dropout, self.initializer_range, self.rms_norm_eps = dropout, initializer_range, rms_norm_eps
        self.use_cache = True
        self.activation_checkpointing = activation_checkpointing
        super().__init__(
            bos_token_id=bos_token_id, eos_token_id=eos_token_id, pad_token_id=pad_token_id,
            tie_word_embeddings=tie_word_embeddings, **kwargs,
        )


class MixedDecoderBlock(nn.Module):
    def __init__(self, config: LegalGQAMoEConfig, layer_idx: int):
        super().__init__()
        self.layer_idx = layer_idx
        self.is_mha = layer_idx in config.mha_layers
        n_kv_head = config.n_head if self.is_mha else config.gqa_n_kv_head
        # GroupedQueryAttention reads n_head/n_kv_head/n_embd/dropout off a config
        # object -- reusing it unmodified means building one small per-layer config
        # with just n_kv_head varied, rather than touching model 1's tested file at all
        attn_config = LegalGQAConfig(
            vocab_size=config.vocab_size, n_positions=config.n_positions,
            n_embd=config.n_embd, n_head=config.n_head, n_kv_head=n_kv_head,
            dropout=config.dropout,
        )
        self.ln_1 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.attn = GroupedQueryAttention(attn_config)
        self.ln_2 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.moe = MoE(config.n_embd, config.n_routed_experts, config.n_shared_experts,
                       config.moe_top_k, config.expert_d_ff, config.moe_aux_loss_coef)

    def forward(self, x, cos, sin, attn_bias, is_causal, past_kv=None, use_cache=False):
        attn_out, present = self.attn(self.ln_1(x), cos, sin, attn_bias, is_causal, past_kv, use_cache)
        x = x + attn_out
        moe_out = self.moe(self.ln_2(x))
        x = x + moe_out
        return x, present, self.moe.last_aux_loss


class LegalGQAMoEForCausalLM(PreTrainedModel):
    config_class = LegalGQAMoEConfig
    _tied_weights_keys = {"lm_head.weight": "wte.weight"}

    def __init__(self, config: LegalGQAMoEConfig):
        super().__init__(config)
        self.head_dim = config.n_embd // config.n_head
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([MixedDecoderBlock(config, i) for i in range(config.n_layer)])
        self.ln_f = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight
        self.post_init()

    def _init_weights(self, module):
        std = self.config.initializer_range
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
        for name, p in module.named_parameters(recurse=False):
            if name in ("o_proj.weight", "down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=std / math.sqrt(2 * self.config.n_layer))

    def get_input_embeddings(self):
        return self.wte

    def set_input_embeddings(self, value):
        self.wte = value

    def get_output_embeddings(self):
        return self.lm_head

    def set_output_embeddings(self, new_embeddings):
        self.lm_head = new_embeddings

    def _rope_slice(self, past_len, T, device, dtype):
        inv_freq = 1.0 / (self.config.rope_theta ** (torch.arange(0, self.head_dim, 2, device=device).float() / self.head_dim))
        pos = torch.arange(past_len, past_len + T, device=device).float()
        emb = torch.cat((torch.outer(pos, inv_freq),) * 2, dim=-1)
        return emb.cos().to(dtype), emb.sin().to(dtype)

    def _build_attn_bias(self, attention_mask, past_len, T, device, dtype):
        no_padding = attention_mask is None or bool(attention_mask.bool().all())
        if no_padding:
            return None, T > 1
        total_len = past_len + T
        pad = attention_mask[:, None, None, :].to(dtype)
        bias = (1.0 - pad) * torch.finfo(dtype).min
        if T > 1:
            q_pos = torch.arange(past_len, past_len + T, device=device).unsqueeze(1)
            k_pos = torch.arange(0, total_len, device=device).unsqueeze(0)
            causal = (k_pos > q_pos).to(dtype) * torch.finfo(dtype).min
            bias = bias + causal[None, None]
        return bias, False

    def forward(self, input_ids, attention_mask=None, past_key_values=None, labels=None, use_cache=None):
        amp = input_ids.is_cuda and (attention_mask is None or bool(attention_mask.bool().all()))
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            return self._forward_impl(input_ids, attention_mask, past_key_values, labels, use_cache)
        # no **kwargs above on purpose -- Trainer treats a forward() accepting **kwargs
        # as self-normalizing its loss, breaking grad-accum scaling (models 1/2/3's note)

    def _forward_impl(self, input_ids, attention_mask, past_key_values, labels, use_cache):
        use_cache = self.config.use_cache if use_cache is None else use_cache
        if self.training:
            use_cache = False
        B, T = input_ids.shape
        past_len = past_key_values[0][0].shape[2] if past_key_values is not None else 0

        x = self.drop(self.wte(input_ids))
        cos, sin = self._rope_slice(past_len, T, input_ids.device, x.dtype)
        attn_bias, is_causal = self._build_attn_bias(attention_mask, past_len, T, input_ids.device, x.dtype)

        presents = [] if use_cache else None
        moe_losses = []
        for block in self.blocks:
            pkv = past_key_values[block.layer_idx] if past_key_values is not None else None
            if self.training and self.config.activation_checkpointing:
                def run(h, blk=block):
                    h2, _, moe_l = blk(h, cos, sin, attn_bias, is_causal, None, False)
                    if moe_l is not None:
                        moe_losses.append(moe_l)
                    return h2
                x = torch_checkpoint(run, x, use_reentrant=False)
            else:
                x, present, moe_l = block(x, cos, sin, attn_bias, is_causal, pkv, use_cache)
                if use_cache:
                    presents.append(present)
                if moe_l is not None:
                    moe_losses.append(moe_l)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), ignore_index=-100,
            )
            if moe_losses:
                loss = loss + torch.stack(moe_losses).mean()

        return CausalLMOutputWithPast(
            loss=loss,
            logits=None if labels is not None else logits,
            past_key_values=tuple(presents) if use_cache else None,
        )

In [ ]:
%%writefile /kaggle/working/generate_utils.py
"""
Drop-in replacement for the starter notebook's generate_batch() in
'## 19. Generate 128 Words'. Same signature and return shape, so cell 20
(the prediction loop) and cells 21-22 (prediction.json / submission.zip)
need no changes at all.

Written as a plain per-step loop against the model's own forward() and its
tuple-based KV cache (see legal_gqa_model.py) rather than model.generate(),
to avoid depending on transformers' internal Cache-class machinery.
"""
import torch
import torch.nn.functional as F


def _top_p_filter(logits, top_p):
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    probs = F.softmax(sorted_logits, dim=-1)
    cum_probs = torch.cumsum(probs, dim=-1)
    # keep the smallest prefix whose cumulative prob exceeds top_p;
    # always keep at least the top-1 token
    remove = cum_probs > top_p
    remove[..., 1:] = remove[..., :-1].clone()
    remove[..., 0] = False
    sorted_logits[remove] = float("-inf")
    out = torch.full_like(logits, float("-inf"))
    out.scatter_(-1, sorted_idx, sorted_logits)
    return out


def _build_zero_progress_mask(tokenizer):
    """Token ids whose decoded text is empty after stripping whitespace --
    not just the registered special tokens (eos/pad/bos, which can coincide,
    as they do for this project's tokenizer), but any other token a
    byte-level BPE vocab learned as a pure newline/space merge (expected from
    real formatted text -- legal documents especially, with clause breaks and
    indentation). Generating one of these consumes a real generation step but
    contributes nothing to the word count at the end, the exact same leak
    mechanism as unblocked EOS, just not caught by checking all_special_ids
    alone. Found via direct inspection, not assumed: 12 such ids exist even
    in a small test vocabulary, so this is a real, not theoretical, gap."""
    ids = [tid for tid in range(len(tokenizer)) if tokenizer.decode([tid]).strip() == ""]
    return torch.tensor(sorted(set(ids)), dtype=torch.long)


def _build_word_token_table(tokenizer):
    """token_id -> lowercased word it produces, only for tokens that start a
    new word (GPT-2 byte-level BPE marks these with a leading space)."""
    table = {}
    for tid in range(len(tokenizer)):
        text = tokenizer.decode([tid])
        if text[:1] in (" ", "\n"):
            word = text.strip().lower()
            if word:
                table.setdefault(word, []).append(tid)
    return {w: torch.tensor(ids) for w, ids in table.items()}


def _verbatim_overlap(prompt_words, continuation_words, min_run=6):
    """True if a >= min_run-word run from the continuation appears verbatim in
    the prompt (case-insensitive). Used only for a post-hoc visibility check --
    the actual prevention happens via trigram seeding below."""
    prompt_joined = " ".join(w.lower() for w in prompt_words)
    for i in range(len(continuation_words) - min_run + 1):
        window = " ".join(w.lower() for w in continuation_words[i:i + min_run])
        if window and window in prompt_joined:
            return window
    return None


@torch.inference_mode()
def generate_batch(
    prompts=None,
    model=None,
    tokenizer=None,
    device=None,
    target_words=128,
    max_new_tokens=384,
    temperature=0.7,
    top_p=0.95,
    tail_window=8,  # tokens looked at when checking for a repeated word
    token_ids=None,  # optional: list of lists of ints (one per prompt) -- skips re-tokenizing
):
    model.eval()
    max_length = model.config.n_positions - max_new_tokens
    if token_ids is not None:
        trunc = [ids[:max_length] for ids in token_ids]  # same right-truncation as the tokenizer path below
        pad_id = tokenizer.pad_token_id
        B = len(trunc)
        maxlen = max(len(ids) for ids in trunc)
        input_ids = torch.full((B, maxlen), pad_id, dtype=torch.long)
        attention_mask = torch.zeros((B, maxlen), dtype=torch.long)
        for i, ids in enumerate(trunc):
            input_ids[i, maxlen - len(ids):] = torch.tensor(ids, dtype=torch.long)  # left-pad, matches tokenizer.padding_side="left"
            attention_mask[i, maxlen - len(ids):] = 1
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
    else:
        inputs = tokenizer(
            prompts, return_tensors="pt", padding=True, truncation=True,
            max_length=max_length,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        input_ids = inputs["input_ids"]
        attention_mask = inputs["attention_mask"]
    input_lengths = attention_mask.sum(dim=1)
    B = input_ids.size(0)

    word_table = getattr(generate_batch, "_word_table_cache", None)
    if word_table is None:
        word_table = _build_word_token_table(tokenizer)
        generate_batch._word_table_cache = word_table  # cache across calls

    zero_progress_ids = getattr(generate_batch, "_zero_progress_cache", None)
    if zero_progress_ids is None:
        zero_progress_ids = _build_zero_progress_mask(tokenizer)
        generate_batch._zero_progress_cache = zero_progress_ids
    zero_progress_ids = zero_progress_ids.to(device)

    # prefill: run the full prompt once to seed the KV cache
    out = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=True)
    past_key_values = out.past_key_values
    next_logits = out.logits[:, -1, :]

    # Real per-batch generation budget, not the fixed max_new_tokens guess -- a fixed
    # cap under-provisions short prompts (which have far more safe room than that) and
    # can still be too little for hard rows where repetition-blocking wastes steps.
    # Bounded by the model's actual trained context length, so this never pushes
    # generation into positions the model was never trained on.
    available_steps = max(1, model.config.n_positions - input_ids.shape[1])

    generated = [[] for _ in range(B)]
    trigram_seen = [set() for _ in range(B)]
    # Seed with the PROMPT's own trigrams too, not just trigrams generated so far --
    # this makes the SAME "block recurrence of any seen trigram" guard also stop the
    # model from reproducing 3+ consecutive tokens verbatim from the prompt (the task's
    # "must not copy the prompt" requirement), reusing an already-tested mechanism
    # rather than adding a new one.
    prompt_ids_per_row = []
    for row in range(B):
        real_ids = input_ids[row][attention_mask[row].bool()].tolist()
        prompt_ids_per_row.append(real_ids)
        for i in range(len(real_ids) - 2):
            trigram_seen[row].add(tuple(real_ids[i:i + 3]))
    finished_word_count = [0] * B
    cur_ids = input_ids
    cur_mask = attention_mask

    for _ in range(available_steps):
        step_logits = next_logits / max(temperature, 1e-5)

        for row in range(B):
            if finished_word_count[row] >= target_words:
                continue

            # --- guard 0: block any token that makes zero word-progress ---
            # Covers EOS (a model trained on EOS-separated documents will
            # sometimes naturally choose it mid-continuation when it "feels" a
            # document should end) AND any other token a byte-level BPE vocab
            # learned as a pure whitespace/newline merge -- both consume a real
            # generation step and get appended to generated[row], but
            # contribute nothing once skip_special_tokens/strip()/split() runs
            # at the end. Confirmed as a real, measured gap, not theoretical:
            # blocking EOS alone took a production shortfall from 74/128 to
            # 86/128, not all the way to 128 -- the remaining gap is exactly
            # these non-special empty-decode tokens, which the narrower EOS-only
            # check never covered.
            if zero_progress_ids.numel() > 0:
                step_logits[row, zero_progress_ids] = float("-inf")

            # --- guard 1: block a 3rd consecutive repeat of the same word ---
            tail_ids = torch.tensor(generated[row][-tail_window:])
            if tail_ids.numel() > 0:
                tail_text = tokenizer.decode(tail_ids, skip_special_tokens=True)
                words = tail_text.split()
                if len(words) >= 2 and words[-1].lower() == words[-2].lower():
                    block_ids = word_table.get(words[-1].lower())
                    if block_ids is not None:
                        step_logits[row, block_ids.to(device)] = float("-inf")

        # --- guard 2: block recurrence of any trigram already generated ---
        # checked only against each row's current top-50 candidates (the
        # tokens sampling would realistically pick anyway) rather than the
        # full vocab, since this runs every generation step.
        for row in range(B):
            if finished_word_count[row] >= target_words or len(generated[row]) < 2:
                continue
            prefix = (generated[row][-2], generated[row][-1])
            seen = trigram_seen[row]
            # only tokens that would recreate a seen trigram get blocked;
            # we look this up lazily per-row since the seen set is small
            for tid in torch.topk(step_logits[row], k=min(50, step_logits.size(-1))).indices.tolist():
                if (prefix[0], prefix[1], tid) in seen:
                    step_logits[row, tid] = float("-inf")

        filtered = _top_p_filter(step_logits, top_p)
        probs = F.softmax(filtered, dim=-1)
        next_tokens = torch.multinomial(probs, num_samples=1).squeeze(-1)

        for row in range(B):
            if finished_word_count[row] >= target_words:
                next_tokens[row] = tokenizer.eos_token_id
                continue
            tok = next_tokens[row].item()
            generated[row].append(tok)
            if len(generated[row]) >= 3:
                trigram_seen[row].add(tuple(generated[row][-3:]))
            text_so_far = tokenizer.decode(generated[row], skip_special_tokens=True)
            finished_word_count[row] = len(text_so_far.split())

        if all(c >= target_words for c in finished_word_count):
            break

        next_col = next_tokens.unsqueeze(1)
        cur_mask = torch.cat([cur_mask, torch.ones(B, 1, dtype=cur_mask.dtype, device=device)], dim=1)
        out = model(input_ids=next_col, attention_mask=cur_mask, past_key_values=past_key_values, use_cache=True)
        past_key_values = out.past_key_values
        next_logits = out.logits[:, -1, :]

    predictions = []
    for row in range(B):
        text = tokenizer.decode(generated[row], skip_special_tokens=True)
        words = text.split()[:target_words]
        if len(words) < target_words:
            # only happens if this row's prompt was long enough that even the FULL
            # available context budget (n_positions - prompt_length) wasn't enough to
            # reach target_words -- a real limit for the longest prompts, not a bug;
            # printed so it's visible rather than silently submitting a short prediction
            print(f"WARNING: row {row} only reached {len(words)}/{target_words} words "
                  f"(prompt used {input_ids.shape[1]} of {model.config.n_positions} context tokens)")

        prompt_text = tokenizer.decode(prompt_ids_per_row[row], skip_special_tokens=True)
        overlap = _verbatim_overlap(prompt_text.split(), words)
        if overlap is not None:
            # trigram seeding above should make this rare -- printed so it's visible
            # rather than silently submitting a continuation that copies the prompt
            print(f"WARNING: row {row} continuation contains a run copied from the "
                  f"prompt: \"{overlap}\"")

        predictions.append(" ".join(words))
    return predictions

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working")
from model import LegalGQAMoEConfig, LegalGQAMoEForCausalLM
from generate_utils import generate_batch

## 7. Load The Trained Model

In [ ]:
ckpt_dirs = find_dirs("model.safetensors")
final_dirs = [d for d in ckpt_dirs if d.rstrip("/").endswith("legal-gqamoe-final") and "checkpoint-" not in d]
if not final_dirs:
    print("No 'legal-gqamoe-final' folder found. Candidates seen:")
    for d in ckpt_dirs:
        print(" ", d)
    raise FileNotFoundError("Attach model 4's training notebook output (Add Input).")
MODEL_DIR = final_dirs[0]
print("Loading model from:", MODEL_DIR)

model = LegalGQAMoEForCausalLM.from_pretrained(MODEL_DIR).to(device).eval()
print(f"Model size: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

## 8. Generate Predictions for the Test Set

Uses `generate_batch` as-is: top-p + temperature sampling, blocks a 3rd consecutive repeat of the same word, blocks trigram recurrence, and -- seeded with the prompt's own trigrams -- prevents the continuation from copying the prompt verbatim (verified in testing: zero copy warnings across 30 trials on a deliberately adversarial, repetitive prompt). Uses the pre-tokenized cache above if attached, otherwise tokenizes from raw text. Guarantees every continuation reaches exactly 128 words when the prompt's length leaves enough context room to do so.

In [ ]:
predictions = {}
items = list(test_data.items())
total = len(items)
t0 = time.time()

for start in range(0, total, GENERATION_BATCH_SIZE):
    batch = items[start:start + GENERATION_BATCH_SIZE]
    ids = [str(eid) for eid, _ in batch]

    if test_token_ids is not None:
        gen_kwargs = {"token_ids": [test_token_ids[eid] for eid in ids]}
    else:
        gen_kwargs = {"prompts": [rec["text"] for _, rec in batch]}

    outs = generate_batch(
        model=model, tokenizer=tokenizer, device=device,
        target_words=TARGET_GENERATION_WORDS, max_new_tokens=MAX_GENERATION_TOKENS,
        temperature=TEMPERATURE, top_p=TOP_P, **gen_kwargs,
    )
    for eid, text in zip(ids, outs):
        predictions[eid] = {"text": text}

    done = start + len(batch)
    elapsed = time.time() - t0
    rate = done / elapsed
    eta = (total - done) / rate if rate > 0 else float("inf")
    print(f"{done}/{total} ({100*done/total:.1f}%) | {elapsed/60:.1f} min elapsed | ETA {eta/60:.1f} min")

## 9. Save prediction.json and Create Submission ZIP

In [ ]:
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
print("Saved:", PREDICTION_PATH)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")
print("Codabench submission created:", SUBMISSION_ZIP)

lens = [len(v["text"].split()) for v in predictions.values()]
print(f"word count: min={min(lens)} max={max(lens)} mean={sum(lens)/len(lens):.1f}")
print("any empty predictions:", sum(1 for l in lens if l == 0))